# 🤖 Phase 3: Generative AI Automated Insights Script
**Objective:** Query the SQLite database (`virat_kohli_analytics.db`), summarize career performance stats, and use the Google Gemini API (`google-genai`) to generate automated executive analytical commentary.

In [4]:
!pip install -U google-genai

  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 4.5 MB/s eta 0:00:00
Using cached anyio-4.14.2-py3-none-any.whl (125 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-win_amd64.whl (2.1 MB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: anyio
    Fou

In [1]:
import sqlite3
import pandas as pd
from google import genai

# 1. Connect to SQLite database and pull summary metrics
conn = sqlite3.connect('virat_kohli_analytics.db')

query_summary = """
SELECT 
    Format,
    SUM(Matches) AS Total_Matches,
    SUM(Innings) AS Total_Innings,
    SUM(Runs) AS Total_Runs,
    ROUND(CAST(SUM(Runs) AS FLOAT) / NULLIF(SUM(Innings), 0), 2) AS Batting_Avg,
    SUM(Hundreds) AS Total_100s,
    SUM(Fifties) AS Total_50s
FROM fact_yearly_performance
GROUP BY Format;
"""

df_summary = pd.read_sql_query(query_summary, conn)
conn.close()

print("--- Data Extracted from SQL for GenAI Prompt ---")
display(df_summary)

--- Data Extracted from SQL for GenAI Prompt ---


,Format,Total_Matches,Total_Innings,Total_Runs,Batting_Avg,Total_100s,Total_50s
0,ODI,311,305,14797,48.51,54,77
1,T20I,125,131,4188,31.97,1,38
2,Test,123,201,9230,45.92,30,31


## Step 2: Initialize Gemini API & Generate Executive Narrative
Pass the structured SQL aggregation table to `gemini-2.5-flash` to generate a professional sports analyst summary.

In [20]:
import os
import sqlite3
import time
import pandas as pd
from google import genai
from google.genai import errors

# 1. Connect to SQLite database and extract summary metrics
conn = sqlite3.connect('virat_kohli_analytics.db')

query_summary = """
SELECT 
    Format,
    SUM(Matches) AS Total_Matches,
    SUM(Innings) AS Total_Innings,
    SUM(Runs) AS Total_Runs,
    ROUND(CAST(SUM(Runs) AS FLOAT) / NULLIF(SUM(Innings), 0), 2) AS Batting_Avg,
    SUM(Hundreds) AS Total_100s,
    SUM(Fifties) AS Total_50s
FROM fact_yearly_performance
GROUP BY Format;
"""

df_summary = pd.read_sql_query(query_summary, conn)
conn.close()

# 2. Initialize Gemini Client
client = genai.Client()

# 3. Format SQL data into string for prompt context
summary_text = df_summary.to_string(index=False)

prompt = f"""
You are a Senior Sports Data Analyst. Analyze the following international career summary data for Virat Kohli:

{summary_text}

Generate a concise 3-paragraph executive summary covering:
1. Overall career dominance and format adaptability.
2. Conversion rate efficiency and milestone highlights.
3. Strategic takeaway for a BI executive dashboard.

Keep the tone highly professional, objective, and data-backed.
"""

# 4. Use 'gemini-flash-latest' alias
model_name = "gemini-flash-latest"

print("Requesting Gemini AI analysis...")

def call_gemini():
    return client.models.generate_content(
        model=model_name,
        contents=prompt,
    )

try:
    response = call_gemini()
except errors.APIError as e:
    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
        print("⚠️ Quota rate limit hit. Pausing for 35 seconds...")
        time.sleep(35)
        response = call_gemini()
    else:
        raise e

print("\n--- 🤖 GEMINI AI GENERATED INSIGHTS ---")
print(response.text)

# 5. Save output to text file
with open("ai_insights_summary.txt", "w") as f:
    f.write(response.text)

print("\n✅ AI insights generated successfully and saved to 'ai_insights_summary.txt'!")

Requesting Gemini AI analysis...

--- 🤖 GEMINI AI GENERATED INSIGHTS ---
Here is the executive summary based on the provided international career dataset:

**1. Career Dominance and Cross-Format Adaptability**
Virat Kohli’s international dataset reveals an elite elite-level aggregate output of 28,215 runs across 559 matches (637 innings), underscoring sustained durability and world-class versatility. His dominance is anchored by his One Day International (ODI) performance, where he accumulated 14,797 runs at a high average of 48.51 across 305 innings. This traditional limited-overs success is complemented by a robust Test record of 9,230 runs at a 45.92 average across 201 innings. Even in the shortest format (T20I), Kohli generated substantial volume with 4,188 runs at an average of 31.97. Maintaining averages above 30.00 across all three distinct formats demonstrates exceptional technical adaptability and operational consistency across varying match conditions and time horizons.

**2.